# GxP-LLM Evaluation Harness

Run baseline eval on base model, or eval on fine-tuned/quantized models.
Logs results to W&B and saves JSON.

In [ ]:
%pip install -q torch==2.5.1 transformers==4.46.3 peft==0.13.2 bitsandbytes==0.45.0 accelerate==0.34.2
%pip install -q sentence-transformers rouge-score nltk litellm wandb

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
import os
import wandb

wandb.login(key=os.environ.get("WANDB_API_KEY"))
wandb.init(project="gxp-llm", name="eval-baseline-qwen2.5-7b", job_type="eval")

In [ ]:
# Configure model to evaluate
MODEL_PATH = "unsloth/Qwen2.5-7B"  # or local path like ./merged_16bit
ADAPTER_PATH = None  # or ./lora_adapter
DATA_DIR = "/kaggle/input/gxp-data"
JUDGE_MODEL = "gpt-4o-mini"  # requires OPENAI_API_KEY in secrets
OUTPUT_DIR = "./eval_results"

In [ ]:
# Run evaluation
import sys
sys.path.insert(0, "/kaggle/working/GxP-LLM")  # adjust if needed

from eval.run import run_full_eval

results = run_full_eval(
    model_path=MODEL_PATH,
    adapter_path=ADAPTER_PATH,
    data_dir=DATA_DIR,
    judge_model=JUDGE_MODEL,
    output_dir=OUTPUT_DIR,
)

In [ ]:
# Log summary to W&B
import json

for split_name, result in results.items():
    wandb.log({f"{split_name}/exact_match": result.metrics.get('exact_match', 0)})
    wandb.log({f"{split_name}/rougeL_f1": result.metrics.get('rougeL', {}).get('rougeL_f1', 0)})
    wandb.log({f"{split_name}/bleu": result.metrics.get('bleu', 0)})
    
    if result.judge_scores:
        for cat, scores in result.judge_scores.items():
            for metric, val in scores.items():
                wandb.log({f"{split_name}/judge/{cat}/{metric}": val})
    
    if result.adversarial:
        wandb.log({f"{split_name}/refusal_rate": result.adversarial['refusal_rate']})
        wandb.log({f"{split_name}/false_compliance_rate": result.adversarial['false_compliance_rate']})
        wandb.log({f"{split_name}/helpful_redirect_rate": result.adversarial['helpful_redirect_rate']})

# Save results as artifact
artifact = wandb.Artifact(f"eval-results-{MODEL_PATH.replace('/', '-')}", type="eval")
artifact.add_dir(OUTPUT_DIR)
wandb.log_artifact(artifact)

wandb.finish()